# Template 00: Project Setup

**Purpose:** One-time setup to export schemas and create column subset files

**Run this notebook:**
- Once at project start
- Whenever data structure changes
- To regenerate column subset files

**Outputs:**
- all_columns_master.csv (all columns from master file)
- all_columns_aux.csv (all columns from aux file)
- columns_to_load_during_dataassembly.csv (subset to load for performance)

In [ ]:
# Parameters (injected by papermill if needed)
config_path = "config/car_coll/v1"

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import yaml
import os
import sys
from pathlib import Path

# Add lib to path and setup environment
sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment, get_machine_id

print("########################################")
print("# PROJECT SETUP")
print("########################################")

# Auto-detect project root (cross-machine compatible)
project_root = setup_notebook_environment()
machine_id = get_machine_id()

print(f"\nProject root: {project_root}")
print(f"Machine ID: PC{machine_id}")
print(f"Current directory: {os.getcwd()}")

In [ ]:
# Load config (with machine-specific paths)
from utils import load_config
config_file = f"{config_path}/config.yaml"
cfg = load_config(config_file)

data_root = cfg['paths']['data_root']
print(f"\nData root: {data_root}")
print(f"Config: {config_path}")

## Step 1: Export Master File Schema

In [ ]:
# Export master file schema
master_path = f"{data_root}/{cfg['data']['master_file']}"
print(f"\nExporting schema from: {master_path}")

# Read schema only (no data)
parquet_file = pq.ParquetFile(master_path)
schema = parquet_file.schema_arrow

# Create dataframe
master_schema_data = []
for name in schema.names:
    master_schema_data.append({
        'column_name': name,
        'dtype': str(schema.field(name).type),
        'notes': ''
    })

master_schema_df = pd.DataFrame(master_schema_data)

# Save
master_schema_file = f"{config_path}/all_columns_master.csv"
master_schema_df.to_csv(master_schema_file, index=False)

print(f"Exported {len(master_schema_df)} columns to: {master_schema_file}")
print(f"\nColumn type distribution:")
print(master_schema_df['dtype'].value_counts().head(10))

## Step 2: Export Aux File Schema

In [ ]:
# Export aux file schema
aux_path = f"{data_root}/{cfg['data']['aux_file']}"
print(f"\nExporting schema from: {aux_path}")

# Read schema only
parquet_file = pq.ParquetFile(aux_path)
schema = parquet_file.schema_arrow

# Create dataframe
aux_schema_data = []
for name in schema.names:
    aux_schema_data.append({
        'column_name': name,
        'dtype': str(schema.field(name).type),
        'notes': ''
    })

aux_schema_df = pd.DataFrame(aux_schema_data)

# Save
aux_schema_file = f"{config_path}/all_columns_aux.csv"
aux_schema_df.to_csv(aux_schema_file, index=False)

print(f"Exported {len(aux_schema_df)} columns to: {aux_schema_file}")
print(f"\nAux columns:")
print(aux_schema_df['column_name'].tolist())

## Step 3: Create Column Subset for Data Assembly

In [ ]:
# Load feature list from columns_inclusion.csv
features_file = f"{config_path}/{cfg['features']['inclusion_file']}"
print(f"\nLoading features from: {features_file}")

features_df = pd.read_csv(features_file, comment='#')
feature_list = features_df['column_name'].tolist()

print(f"Features to include: {len(feature_list)}")

In [ ]:
# Essential columns (always needed)
essential_cols = [
    cfg['data']['join_key'],  # Join key (e.g., vin_date)
    'vin',                     # Vehicle identifier
]

print(f"\nEssential columns: {essential_cols}")

In [ ]:
# Combine and create subset file
columns_to_load = essential_cols + feature_list

# Check which columns exist in master file
master_cols_set = set(master_schema_df['column_name'])
missing_cols = [c for c in columns_to_load if c not in master_cols_set]

if missing_cols:
    print(f"\nWARNING: {len(missing_cols)} columns not found in master file:")
    for col in missing_cols[:10]:  # Show first 10
        print(f"  - {col}")
    if len(missing_cols) > 10:
        print(f"  ... and {len(missing_cols) - 10} more")
    
    # Filter to only existing columns
    columns_to_load = [c for c in columns_to_load if c in master_cols_set]
    print(f"\nFiltered to {len(columns_to_load)} existing columns")
else:
    print(f"\nAll {len(columns_to_load)} columns found in master file")

In [ ]:
# Create output dataframe
output_data = []
for col in columns_to_load:
    source = 'essential' if col in essential_cols else 'feature'
    output_data.append({
        'column_name': col,
        'source': source,
        'notes': ''
    })

output_df = pd.DataFrame(output_data)

# Save
subset_file = f"{config_path}/columns_to_load_during_dataassembly.csv"
output_df.to_csv(subset_file, index=False)

print(f"\nCreated: {subset_file}")
print(f"  Essential columns: {sum(output_df['source'] == 'essential')}")
print(f"  Feature columns: {sum(output_df['source'] == 'feature')}")
print(f"  Total columns: {len(output_df)}")

## Summary

In [ ]:
# STEP: Auto-detect object→numeric conversions
print("\n" + "="*60)
print("STEP: DETECTING TYPE CONVERSIONS")
print("="*60)

conversions = []

# Check each column in master dataset
for col in master.columns:
    if master[col].dtype == 'object':
        # Try numeric conversion
        test_convert = pd.to_numeric(master[col], errors='coerce')
        success_rate = test_convert.notna().sum() / len(test_convert)
        
        # If >50% convert successfully, recommend conversion
        if success_rate > 0.5:
            conversions.append({
                'column_name': col,
                'source_dtype': 'object',
                'target_dtype': 'numeric',
                'success_rate': f"{success_rate:.1%}"
            })

# Save to config
if conversions:
    conv_df = pd.DataFrame(conversions)
    conv_file = f"{config_path}/type_conversions.csv"
    conv_df.to_csv(conv_file, index=False)
    print(f"\n✓ Created type_conversions.csv ({len(conversions)} columns)")
    print(f"\nColumns to convert:")
    for _, row in conv_df.iterrows():
        print(f"  • {row['column_name']} (success: {row['success_rate']})")
    print(f"\n⚠️  IMPORTANT: Review {conv_file} before running pipeline!")
    print(f"   This file will be used in Template 02 to convert data types.")
else:
    print(f"\n✓ No object→numeric conversions needed")

In [ ]:
print("\n" + "="*60)
print("SETUP SUMMARY")
print("="*60)

total_master_cols = len(master_schema_df)
cols_to_load = len(output_df)
cols_skipped = total_master_cols - cols_to_load
reduction_pct = (cols_skipped / total_master_cols) * 100

print(f"\nMaster file columns: {total_master_cols}")
print(f"Columns to load: {cols_to_load}")
print(f"Columns skipped: {cols_skipped}")
print(f"Memory reduction: {reduction_pct:.1f}%")

print(f"\nFiles created:")
print(f"  - {master_schema_file}")
print(f"  - {aux_schema_file}")
print(f"  - {subset_file}")

print(f"\nNext steps:")
print(f"  1. Review {subset_file}")
print(f"  2. Edit to add/remove columns if needed")
print(f"  3. Run Template 01 (data assembly)")

print("\n" + "="*60)
print("SETUP COMPLETE")
print("="*60)